# Building a Custom Control/Policy

This notebook walks through building a **custom control** from scratch — a validator that automatically checks every LLM response before it leaves the pipeline.

**What we'll build:** A `TonePolicyControl` that enforces configurable response policies using an LLM-as-judge. You define rules like *"use a professional tone"* or *"do not speculate"*, and the control checks every response against them.

**What you'll learn:**
- The `BaseControl` interface (3 required things)
- How to implement `check()` with real validation logic
- Wiring a custom control into the attestation pipeline
- Reading validation results from the attestation evidence
- Flag mode vs. block mode
- Deploying via `glacis.yaml`

```bash
pip install glacis openai
```

Set `OPENAI_API_KEY` in your environment or in `tests/.env`.

In [1]:
import json, os, sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Load .env
env_path = REPO_ROOT / "tests" / ".env"
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
SIGNING_SEED = b"demo-custom-ctrl-seed-0000000000"  # 32 bytes

# Evidence storage — must match demos/glacis.yaml so get_evidence() finds results
STORAGE_BACKEND = "json"
STORAGE_PATH = "~/.glacis"

print("Setup complete")

Setup complete


## 1. The BaseControl Interface

Every Glacis control — built-in or custom — implements the same interface. Let's look at what's required:

In [2]:
from glacis.controls.base import BaseControl, ControlResult
import inspect

# The entire public interface:
for name, method in inspect.getmembers(BaseControl, predicate=inspect.isfunction):
    if not name.startswith("_"):
        sig = inspect.signature(method)
        print(f"  {name}{sig}")

print("\nTo build a custom control you need exactly 3 things:")
print("  1. Set a 'control_type' class attribute (a string name)")
print("  2. Implement check(text: str) -> ControlResult")
print("  3. Return a ControlResult with your detection info")
print("\nThe close() method is optional — override it if you hold expensive resources.")

  check(self, text: str) -> glacis.controls.base.ControlResult
  close(self) -> None

To build a custom control you need exactly 3 things:
  1. Set a 'control_type' class attribute (a string name)
  2. Implement check(text: str) -> ControlResult
  3. Return a ControlResult with your detection info

The close() method is optional — override it if you hold expensive resources.


### ControlResult Fields

Every control returns a `ControlResult`. Here are the key fields:

| Field | Type | Description |
|-------|------|-------------|
| `control_type` | `str` | Your control's name (must match class attribute) |
| `detected` | `bool` | Was an issue found? |
| `action` | `str` | `"forward"` (pass), `"flag"` (warn), `"block"` (halt), or `"error"` |
| `score` | `float \| None` | Confidence score (>= 0, scale is control-specific) |
| `categories` | `list[str]` | What was detected (e.g., `["tone_violation"]`) |
| `latency_ms` | `int` | How long the check took |
| `metadata` | `dict` | Anything else — goes into the attestation audit trail |

## 2. Build the Control

We'll build a `TonePolicyControl` that checks every LLM response against a list of configurable policy rules. The control:

1. **Sends** the response + your policy rules to a judge LLM
2. **Gets back** a pass/fail verdict for each rule with rationale
3. **Flags or blocks** if any rules are violated

This is a broadly useful pattern — using one LLM to validate another's output against your organization's policies.

In [ ]:
import json
import time
import logging
from typing import Any

from glacis.controls.base import BaseControl, ControlResult

logger = logging.getLogger(__name__)


class TonePolicyControl(BaseControl):
    """Enforce tone and policy rules on LLM responses using an LLM judge.

    Sends each response to a judge model along with your policy rules.
    If any rule is violated, the control flags or blocks the response.
    The per-rule verdicts and rationale are recorded in the attestation.
    """

    # --- Required thing #1: control_type ---
    # This names your control in attestation records.
    # Glacis recognizes 8 types: pii, jailbreak, word_filter,
    #   content_safety, topic, prompt_security, grounding, custom
    # Unknown types silently map to "custom" in the attestation.
    control_type = "content_safety"

    def __init__(
        self,
        openai_api_key: str,
        policies: list[str],
        if_detected: str = "flag",
        model: str = "gpt-4o-mini",
    ) -> None:
        from openai import OpenAI

        self._client = OpenAI(api_key=openai_api_key)
        self._policies = policies
        self._action = if_detected
        self._model = model

    # --- Required thing #2: implement check() ---
    def check(self, text: str) -> ControlResult:
        """Check the LLM response against all policy rules."""
        start = time.perf_counter()

        # Build the numbered rules list for the judge prompt
        rules_text = "\n".join(
            f"{i+1}. {rule}" for i, rule in enumerate(self._policies)
        )

        prompt = f"""You are a policy compliance judge. Evaluate whether the following
                    response violates ANY of the listed rules.

                    Rules:
                    {rules_text}

                    Response to evaluate:
                    {text}

                    For each rule, respond with JSON only:
                    {{
                    "results": [
                        {{"rule": 1, "passed": true/false, "rationale": "brief explanation"}},
                        ...
                    ]
                    }}"""

        try:
            response = self._client.chat.completions.create(
                model=self._model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=500,
            )
            raw = response.choices[0].message.content or ""
            verdicts = self._parse_verdicts(raw)
        except Exception as e:
            logger.warning("Tone policy check failed: %s", e)
            latency_ms = int((time.perf_counter() - start) * 1000)
            return ControlResult(
                control_type=self.control_type,
                detected=False,
                action="error",
                latency_ms=latency_ms,
                metadata={"error": str(e)},
            )

        # Compute results
        violations = [v for v in verdicts if not v.get("passed", True)]
        has_violations = len(violations) > 0
        pass_rate = (len(verdicts) - len(violations)) / max(len(verdicts), 1)
        latency_ms = int((time.perf_counter() - start) * 1000)

        categories = []
        for v in violations:
            rule_idx = v.get("rule", "?")
            if isinstance(rule_idx, int) and 1 <= rule_idx <= len(self._policies):
                categories.append(self._policies[rule_idx - 1][:50])
            else:
                categories.append(f"rule_{rule_idx}")

        # --- Required thing #3: return ControlResult ---
        return ControlResult(
            control_type=self.control_type,
            detected=has_violations,
            action=self._action if has_violations else "forward",
            score=round(pass_rate, 2),
            categories=categories[:10],
            latency_ms=latency_ms,
            metadata={
                "num_rules": len(self._policies),
                "num_violations": len(violations),
                "pass_rate": round(pass_rate, 2),
                "verdicts": verdicts,
                "model": self._model,
            },
        )

    def _parse_verdicts(self, raw: str) -> list[dict[str, Any]]:
        """Parse the judge's JSON verdicts from the raw response."""
        cleaned = raw.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned[3:]
            if cleaned.endswith("```"):
                cleaned = cleaned[:-3]
            cleaned = cleaned.strip()

        try:
            data = json.loads(cleaned)
            return data.get("results", [])
        except (json.JSONDecodeError, AttributeError):
            return []

    def close(self) -> None:
        """Release the OpenAI client."""
        if hasattr(self._client, "close"):
            self._client.close()


print(f"Control class defined: {TonePolicyControl.control_type}")
print(f"Extends BaseControl: {issubclass(TonePolicyControl, BaseControl)}")

Control class defined: content_safety
Extends BaseControl: True


## 3. Define Your Policies

Policies are plain-English rules. The judge LLM evaluates each one independently. You can add, remove, or change rules without modifying any code.

In [4]:
POLICIES = [
    "Use a professional, neutral tone throughout",
    "Do not speculate or make claims without factual basis",
    "Do not mention competitor products or services by name",
    "Do not provide medical, legal, or financial advice",
]

tone_control = TonePolicyControl(
    openai_api_key=OPENAI_API_KEY,
    policies=POLICIES,
    if_detected="flag",   # "flag" = continue but record; "block" = halt
    model="gpt-4o-mini",
)

print(f"Control: {tone_control.control_type}")
print(f"Policies: {len(POLICIES)} rules")
for i, p in enumerate(POLICIES, 1):
    print(f"  {i}. {p}")

Control: content_safety
Policies: 4 rules
  1. Use a professional, neutral tone throughout
  2. Do not speculate or make claims without factual basis
  3. Do not mention competitor products or services by name
  4. Do not provide medical, legal, or financial advice


## 4. Wire It Into the Attestation Pipeline

Pass the control via `output_controls=`. Every LLM response is automatically checked — the control runs in the attestation pipeline alongside any built-in controls.

In [5]:
from glacis.integrations.openai import attested_openai, get_last_receipt
from glacis.integrations.base import get_evidence

client = attested_openai(
    openai_api_key=OPENAI_API_KEY,
    offline=True,
    signing_seed=SIGNING_SEED,
    output_controls=[tone_control],
)

print("Attested client created with tone/policy output control")
print("Every LLM response will be checked against your policy rules.")

Attested client created with tone/policy output control
Every LLM response will be checked against your policy rules.


## 5. Test It — Clean Response (Pass)

Ask a straightforward question. The response should pass all policy rules.

In [6]:
print("Sending a normal query (should pass all policies)...\n")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content":
        "What are the key benefits of using encryption at rest for data storage?"
    }],
    temperature=0.3,
)

print("LLM Response:")
print(response.choices[0].message.content)

receipt = get_last_receipt()
print(f"\nAttestation: {receipt.id}")
print(f"Status: {receipt.witness_status}")

Sending a normal query (should pass all policies)...

LLM Response:
Encryption at rest refers to the practice of encrypting data that is stored on physical media, such as hard drives, SSDs, or cloud storage. Here are some key benefits of using encryption at rest for data storage:

1. **Data Protection**: Encryption at rest protects sensitive data from unauthorized access. Even if an attacker gains physical access to the storage medium, they cannot read the data without the encryption keys.

2. **Compliance**: Many regulations and standards (such as GDPR, HIPAA, PCI-DSS) require organizations to protect sensitive data. Encryption at rest helps organizations meet these legal and regulatory requirements, avoiding potential fines and penalties.

3. **Data Breach Mitigation**: In the event of a data breach, encrypted data is much less valuable to attackers. Without the decryption keys, the stolen data is rendered useless, reducing the impact of a breach.

4. **Confidentiality**: Encryption 

## 6. Inspect the Attestation Evidence

The attestation evidence includes `control_plane_results` with the per-rule verdicts and pass/fail decision — all cryptographically recorded.

In [7]:
evidence = get_evidence(receipt.id, storage_backend=STORAGE_BACKEND, storage_path=STORAGE_PATH)

cp = evidence.get("control_plane_results", {})
print("Control Plane Results:")
print(json.dumps(cp, indent=2, default=str))

Control Plane Results:
{
  "policy": {
    "id": "hipaa-safe-harbor",
    "version": "1.0",
    "model": {
      "model_id": "gpt-4o-mini",
      "provider": "openai",
      "system_prompt_hash": null,
      "temperature": 0.3
    },
    "environment": "development",
    "tags": [
      "healthcare",
      "hipaa",
      "demo"
    ]
  },
  "determination": {
    "action": "forwarded"
  },
  "controls": [
    {
      "id": "glacis-input-word_filter",
      "type": "word_filter",
      "version": "0.6.0",
      "provider": "glacis",
      "latency_ms": 0,
      "status": "forward",
      "score": null,
      "result_hash": null,
      "stage": "input"
    },
    {
      "id": "glacis-input-pii",
      "type": "pii",
      "version": "0.6.0",
      "provider": "glacis",
      "latency_ms": 20,
      "status": "forward",
      "score": null,
      "result_hash": null,
      "stage": "input"
    },
    {
      "id": "glacis-input-jailbreak",
      "type": "jailbreak",
      "version": "0.6

In [8]:
# ControlPlaneResults: {"controls": [...], "determination": {...}, "policy": {...}}
# Each control is a ControlExecution with: type, status, score, stage, latency_ms
all_controls = cp.get("controls", [])
output_ctrls = [c for c in all_controls if c.get("stage") == "output"]

print("Output Control Results:\n")
for ctrl in output_ctrls:
    print(f"  Control Type: {ctrl.get('type', 'unknown')}")
    print(f"  Status: {ctrl.get('status', 'unknown')}")
    print(f"  Score (pass rate): {ctrl.get('score', 'N/A')}")
    print(f"  Latency: {ctrl.get('latency_ms', 0)}ms")
    print()

determination = cp.get("determination", {})
print(f"Overall Determination: {determination.get('action', 'unknown')}")

Output Control Results:

  Control Type: pii
  Status: forward
  Score (pass rate): None
  Latency: 149ms

  Control Type: content_safety
  Status: forward
  Score (pass rate): 1.0
  Latency: 2633ms

Overall Determination: forwarded


## 7. Test It — Policy Violation (Flag)

Now let's craft a prompt that's likely to produce a response violating some of our policies.

In [9]:
print("Sending a prompt designed to trigger policy violations...\n")

bad_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content":
        "Write an opinionated rant about why our product is way better than "
        "Datadog and Splunk. Be aggressive and speculative about their future. "
        "Also recommend some stocks to buy."
    }],
    temperature=0.7,
)

print("LLM Response:")
print(bad_response.choices[0].message.content)

bad_receipt = get_last_receipt()
print(f"\nAttestation: {bad_receipt.id}")

Sending a prompt designed to trigger policy violations...

LLM Response:
Alright, let’s dive right in. When it comes to monitoring and observability tools, it’s time to face the facts: our product absolutely obliterates Datadog and Splunk. Seriously, if you’re still using either of those relics, it's like clinging to a flip phone in the era of smartphones. Here’s why we’re not just better—we’re the future.

First off, let’s talk about **performance**. Datadog and Splunk may have their moments, but they’re bogged down by complex pricing structures and bloated feature sets that often lead to analysis paralysis. Our product is streamlined, intuitive, and built for speed. While they’re busy trying to justify their exorbitant pricing, we’re delivering unmatched performance at a fraction of the cost. Why pay for a luxury yacht when you can have a top-tier speedboat that gets you to the same destination faster?

Now, let’s address the elephant in the room: **innovation**. Datadog and Splunk a

In [10]:
bad_evidence = get_evidence(bad_receipt.id, storage_backend=STORAGE_BACKEND, storage_path=STORAGE_PATH)
bad_cp = bad_evidence.get("control_plane_results", {})
bad_controls = bad_cp.get("controls", [])
bad_output_ctrls = [c for c in bad_controls if c.get("stage") == "output"]

print("Output Control Results:\n")
for ctrl in bad_output_ctrls:
    status = ctrl.get('status', 'unknown')
    flagged = status in ("flag", "block")
    label = "FLAGGED" if flagged else "PASSED"
    print(f"  [{label}] {ctrl.get('type', 'unknown')}")
    print(f"  Status: {status}")
    print(f"  Score (pass rate): {ctrl.get('score', 'N/A')}")
    print()

bad_determination = bad_cp.get("determination", {})
print(f"Overall Determination: {bad_determination.get('action', 'unknown')}")
print(f"\nBoth responses — clean and violating — have the policy check result")
print(f"cryptographically recorded in the attestation. This is the audit trail.")

Output Control Results:

  [PASSED] pii
  Status: forward
  Score (pass rate): None

  [FLAGGED] content_safety
  Status: flag
  Score (pass rate): 0.0

Overall Determination: forwarded

Both responses — clean and violating — have the policy check result
cryptographically recorded in the attestation. This is the audit trail.


## 8. Block Mode

Change `if_detected` to `"block"` to halt the pipeline when any policy is violated. A `GlacisBlockedError` is raised, but an attestation is still created recording the block decision.

In [11]:
from glacis.integrations.base import GlacisBlockedError

strict_control = TonePolicyControl(
    openai_api_key=OPENAI_API_KEY,
    policies=POLICIES,
    if_detected="block",  # Block instead of flag
    model="gpt-4o-mini",
)

strict_client = attested_openai(
    openai_api_key=OPENAI_API_KEY,
    offline=True,
    signing_seed=SIGNING_SEED,
    output_controls=[strict_control],
)

print("Sending a policy-violating prompt with BLOCK mode...\n")
try:
    strict_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content":
            "Write an opinionated rant about why our product is way better than "
            "Datadog and Splunk. Be aggressive and speculative about their future."
        }],
        temperature=0.7,
    )
    print("Response passed all policies")
except GlacisBlockedError as e:
    print(f"BLOCKED by control: {e.control_type}")
    if e.score is not None:
        print(f"  Pass rate: {e.score}")
    print(f"\n  The LLM call completed, but the pipeline blocked the response.")
    print(f"  An attestation was still created recording the block decision.")

    blocked_receipt = get_last_receipt()
    if blocked_receipt:
        print(f"\n  Attestation: {blocked_receipt.id}")

Sending a policy-violating prompt with BLOCK mode...

BLOCKED by control: content_safety
  Pass rate: 0.25

  The LLM call completed, but the pipeline blocked the response.
  An attestation was still created recording the block decision.

  Attestation: oatt_994dfc5d-5bcb-4091-bf82-c08dfac8371e


## 9. Deploying via glacis.yaml

Once your control is working, save it to a `.py` file and register it in `glacis.yaml` so it loads automatically — no code changes needed to enable, disable, or tune it.

**Step 1:** Save your control class to a file (e.g., `tone_policy_control.py`) next to `glacis.yaml`.

**Step 2:** Add it to the `custom` section:

```yaml
controls:
  output:
    custom:
      - path: "tone_policy_control.TonePolicyControl"
        enabled: true
        if_detected: "flag"
        args:
          openai_api_key: "${OPENAI_API_KEY}"
          policies:
            - "Use a professional, neutral tone"
            - "Do not speculate or make unsupported claims"
            - "Do not mention competitor products by name"
          model: "gpt-4o-mini"
```

**Step 3:** Load from YAML — the control is discovered and instantiated automatically:

```python
client = attested_openai(
    openai_api_key=OPENAI_API_KEY,
    config="glacis.yaml",
)
```

**Key benefits of YAML config:**
- Enable/disable by flipping `enabled: true/false` — no code changes
- Add or remove policy rules without redeploying
- `${ENV_VAR}` substitution keeps secrets out of config files
- Glacis auto-adds the YAML file's directory to `sys.path` for module resolution

## Summary

To build a custom control:

```python
from glacis.controls.base import BaseControl, ControlResult

class MyControl(BaseControl):
    control_type = "content_safety"        # 1. Name it

    def check(self, text: str) -> ControlResult:  # 2. Implement check()
        # Your logic here — LLM call, regex, ML model, API, anything
        return ControlResult(               # 3. Return ControlResult
            control_type=self.control_type,
            detected=is_bad,
            action="flag" if is_bad else "forward",
            score=confidence,
            metadata={"details": "..."},
        )
```

Wire it in programmatically:

```python
client = attested_openai(
    output_controls=[MyControl()],
)
```

Or deploy via `glacis.yaml`:

```yaml
controls:
  output:
    custom:
      - path: "my_module.MyControl"
        enabled: true
        if_detected: "flag"
        args:
          api_key: "${MY_API_KEY}"
```

**Key properties:**
- Controls run automatically on every LLM call
- Multiple controls run in parallel (latency = max, not sum)
- Errors in controls don't crash the pipeline
- All control decisions are cryptographically attested
- Enable/disable from YAML without touching code
- Works with any integration: OpenAI, Anthropic, Gemini, LiteLLM